# Phase 1: Data Acquisition & Preprocessing (QC)
### Project: Pharmaco-OCD Clinical Decision Support System
**Student:** Final Year Undergraduate, Technology Management  
**Milestone 1:** Harmonised Dataset & Quality Control Report (End of Month 1)

This notebook runs seamlessly **locally** and in **Google Colab**, automating:
1. **Multi-Cohort Acquisition:** Downloading verified *Homo sapiens* transcriptomic cohorts and GPL platform tables from NCBI GEO:
   * `GSE60190` (`GPL6947`, $n = 133$): Postmortem DLPFC OCD, Eating Disorder, and Control cohort.
   * `GSE149980` (`GPL17077`, $n = 68$): Citalopram (SSRI) response in patient-derived LCLs (Responder vs. Non-Responder).
   * `GSE83386` (`GPL17077`, $n = 20$): Fluoxetine (SSRI) response in patient-derived LCLs (Responder vs. Non-Responder).
   * `GSE76826` (`GPL17077`, $n = 32$): Peripheral blood leukocyte MDD, Remission, and Healthy control cohort.
   *(Note: `GSE25974` and `GSE49667` were audited and excluded because they correspond to mouse embryonic tissue and Treg DNA methylation, respectively.)*
2. **Quality Control & $\log_2$ Normalization:** Auditing sample distributions, transforming raw intensities, and flagging inter-sample correlation outliers.
3. **Probe-to-Gene Mapping:** Collapsing Illumina (`Symbol`) and Agilent (`GENE_SYMBOL`) probes to HUGO Gene Symbols.
4. **Cross-Cohort Harmonization & PGC Verification:** Aligning shared genes across platforms, neutralizing batch offsets via Z-score standardization, and inspecting PGC candidate biomarkers.

In [ ]:
# Runtime dependency check (installs only missing packages in Colab / ephemeral environments)
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
    "scipy": "scipy>=1.10.0",
    "statsmodels": "statsmodels>=0.14.0",
    "sklearn": "scikit-learn>=1.3.0",
    "matplotlib": "matplotlib>=3.7.0",
    "seaborn": "seaborn>=0.12.0",
    "GEOparse": "GEOparse>=2.0.3",
    "six": "six>=1.16.0",
    "Bio": "biopython>=1.81",
    "gseapy": "gseapy>=1.0.4",
}

missing = [pkg_spec for mod, pkg_spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

In [ ]:
import os
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", context="notebook")

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO_URL = "https://github.com/disulamihisara123/pharmaco-ocd.git"

if IN_COLAB:
    # Set USE_GOOGLE_DRIVE = True if you store pharmaco-ocd in Google Drive
    USE_GOOGLE_DRIVE = False
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = "/content/drive/MyDrive/pharmaco-ocd"
    else:
        PROJECT_ROOT = "/content/pharmaco-ocd"
        if not os.path.exists(os.path.join(PROJECT_ROOT, "src", "data_loader.py")):
            print(f"Cloning {GITHUB_REPO_URL} into {PROJECT_ROOT}...")
            subprocess.check_call(["git", "clone", "-q", GITHUB_REPO_URL, PROJECT_ROOT])
else:
    cwd = os.getcwd()
    PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if not os.path.exists(os.path.join(SRC_DIR, "data_loader.py")):
    raise FileNotFoundError(f"Could not find src/data_loader.py under {PROJECT_ROOT}.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from data_loader import (
    download_file,
    download_platform_annotation,
    get_geo_series_url,
    parse_series_matrix,
)
from preprocess import (
    check_and_apply_log2,
    clean_missing_probes,
    compute_sample_qc_metrics,
    filter_low_variance_probes,
    harmonize_cohorts,
    map_probes_to_genes,
)

DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DATA_META_DIR = os.path.join(PROJECT_ROOT, "data", "metadata")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "reports", "figures")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_META_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Environment initialized ({'Google Colab' if IN_COLAB else 'Local Python'}).")
print(f"Project Root: {PROJECT_ROOT}")

## 1. Multi-Cohort Download & Parsing (NCBI GEO)
Download the target series matrix files (`GSE60190`, `GSE149980`, `GSE83386`, `GSE76826`) and their corresponding platform annotation tables (`GPL6947`, `GPL17077`). Repeated `!Sample_characteristics_ch1` fields are parsed into named clinical columns.

In [ ]:
COHORTS = {
    "GSE60190": {"role": "OCD Brain Cohort (DLPFC)", "gpl": "GPL6947"},
    "GSE149980": {"role": "SSRI Citalopram Response (LCLs)", "gpl": "GPL17077"},
    "GSE83386": {"role": "SSRI Fluoxetine Response (LCLs)", "gpl": "GPL17077"},
    "GSE76826": {"role": "MDD Remission Blood Cohort", "gpl": "GPL17077"},
}

gpl_paths = {
    gpl: download_platform_annotation(gpl, DATA_RAW_DIR)
    for gpl in sorted({info["gpl"] for info in COHORTS.values()})
}

raw_exprs = {}
metas = {}
summary_rows = []

for gse_id, info in COHORTS.items():
    dest = os.path.join(DATA_RAW_DIR, f"{gse_id}_series_matrix.txt.gz")
    download_file(get_geo_series_url(gse_id), dest)
    df_expr, df_meta = parse_series_matrix(dest)
    raw_exprs[gse_id] = df_expr
    metas[gse_id] = df_meta
    summary_rows.append({
        "Cohort": gse_id,
        "Role": info["role"],
        "Platform": info["gpl"],
        "Probes": df_expr.shape[0],
        "Samples": df_expr.shape[1],
        "Metadata_Cols": df_meta.shape[1],
    })

df_cohort_summary = pd.DataFrame(summary_rows)
display(df_cohort_summary)

### Cohort Acquisition Summary
All four *Homo sapiens* GEO series matrices and both microarray platform tables (`GPL6947` Illumina HumanHT-12 V3.0 and `GPL17077` Agilent SurePrint G3 Human GE v2) loaded cleanly:
* `GSE60190`: 48,679 probes $\times$ 133 DLPFC samples (42 metadata attributes including `ocd`, `dx`, `rin`, `ph`, `pmi`, `age`, `Sex`, `batch1`).
* `GSE149980`: 50,739 probes $\times$ 68 LCL samples (citalopram SSRI treatment with `response status`).
* `GSE83386`: 50,737 probes $\times$ 20 LCL samples (fluoxetine SSRI treatment with `clinical treatment outcome`).
* `GSE76826`: 50,739 probes $\times$ 32 leukocyte samples (Healthy, MDD Depression, MDD Remission).

Next, we audit the clinical target class distributions and repeated-donor structure to prevent data leakage during downstream modeling.

In [ ]:
print("=== GSE60190: OCD vs. Detailed Clinical Diagnosis (dx) ===")
display(pd.crosstab(metas["GSE60190"]["ocd"], metas["GSE60190"]["dx"]))

print("\n=== GSE149980: SSRI Citalopram Response Status ===")
display(metas["GSE149980"]["response status"].value_counts().to_frame("Sample_Count"))

print("\n=== GSE83386: Treatment Arm vs. Clinical Outcome ===")
display(pd.crosstab(metas["GSE83386"]["treatment"], metas["GSE83386"]["clinical treatment outcome"]))

print("\n=== GSE76826: Diagnosis vs. Remission State ===")
display(pd.crosstab(metas["GSE76826"]["diagnosis"], metas["GSE76826"]["state"]))

### Clinical Label Audit Findings
* **`GSE60190`:** Contains 102 pure controls (`dx == Control`), 16 OCD cases (6 pure OCD, 4 Bipolar+OCD, 3 MDD+OCD, 3 Tics+OCD), and 15 Eating Disorder (ED) cases.
* **`GSE149980`:** Balanced SSRI outcome labels (**36 responder**, **32 non-responder**) across 68 samples derived from 17 LCL donors (2 treatments $\times$ 2 timepoints per donor). Downstream ML splits **must** group by donor ID (`StratifiedGroupKFold`) to avoid sample-level leakage.
* **`GSE83386`:** 20 samples from 10 donor cell lines (`LCL1`–`LCL10`) assayed under control vs. fluoxetine (**6 responder**, **4 non-responder** per arm).
* **`GSE76826`:** 12 Healthy controls, 10 active MDD, and 10 remitted MDD blood samples.

## 2. Missing Probe Cleanup & $\log_2$ Scale Normalization
We drop probes with $>20\%$ missingness, impute sparse NaNs with probe medians, verify expression scales via `check_and_apply_log2`, and plot per-sample distributions.

In [ ]:
norm_exprs = {}
for gse_id, df_raw in raw_exprs.items():
    print(f"\n--- {gse_id} ---")
    df_clean = clean_missing_probes(df_raw, max_missing_rate=0.20)
    norm_exprs[gse_id] = check_and_apply_log2(df_clean)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (gse_id, df_n) in zip(axes.flat, norm_exprs.items()):
    subset = df_n.iloc[:, :min(12, df_n.shape[1])]
    sns.boxplot(data=subset, ax=ax, color="steelblue", fliersize=1.5)
    ax.set_title(f"{gse_id}: {COHORTS[gse_id]['role']} (first {subset.shape[1]} samples)")
    ax.set_ylabel("log2 Expression")
    ax.tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
fig_norm_path = os.path.join(FIGURES_DIR, "m1_normalization_distributions.png")
plt.savefig(fig_norm_path, dpi=150)
plt.show()
print(f"Saved figure: {fig_norm_path}")

### Normalization Analysis
* `GSE60190` ($\text{Min}=7.37$, $\text{Max}=15.35$, $P_{99}=12.33$), `GSE149980` ($\text{Min}=-15.78$, $\text{Max}=17.25$, $P_{99}=2.42$), and `GSE76826` ($\text{Min}=-6.93$, $\text{Max}=11.04$, $P_{99}=7.12$) were verified as already $\log_2$-scaled (or $\log_2$-ratio normalized).
* `GSE83386` contained raw unlogged intensities ($\text{Min}=2.39$, $\text{Max}=827{,}061.30$, $P_{99}=84{,}300.26$) and was automatically transformed to $\log_2(x + 1)$ scale.

## 3. Sample-Level Quality Control & Outlier Detection
Compute per-sample median, IQR, missing rate, and mean inter-sample Pearson correlation ($r_i$), flagging potential technical outlier arrays with $z(r_i) < -2.5$.

In [ ]:
qc_tables = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, (gse_id, df_n) in zip(axes.flat, norm_exprs.items()):
    qc_df = compute_sample_qc_metrics(df_n, z_threshold=2.5)
    qc_tables[gse_id] = qc_df
    flagged = qc_df[qc_df["qc_flag"]]
    print(f"{gse_id}: {len(qc_df)} samples | Flagged outliers: {len(flagged)} {list(flagged.index)}")

    colors = ["#d62728" if flag else "#1f77b4" for flag in qc_df["qc_flag"]]
    ax.scatter(range(len(qc_df)), qc_df["mean_sample_corr"], c=colors, s=35, alpha=0.85)
    threshold_line = qc_df["mean_sample_corr"].mean() - 2.5 * qc_df["mean_sample_corr"].std()
    ax.axhline(threshold_line, color="red", linestyle="--", linewidth=1, label="-2.5 SD threshold")
    ax.set_title(f"{gse_id} Sample QC (Flagged={len(flagged)}/{len(qc_df)})")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Mean Inter-Sample Pearson r")
    ax.legend(fontsize=8)

plt.tight_layout()
fig_qc_path = os.path.join(FIGURES_DIR, "m1_sample_qc_correlations.png")
plt.savefig(fig_qc_path, dpi=150)
plt.show()
print(f"Saved figure: {fig_qc_path}")

### Sample QC Findings
* **`GSE60190`:** 4 out of 133 samples (`GSM1467319`, `GSM1467357`, `GSM1467368`, `GSM1467402`) fell below the $-2.5\text{ SD}$ inter-sample correlation threshold.
* **`GSE76826`:** 2 out of 32 samples (`GSM2038465`, `GSM2038478`) were flagged as correlation outliers.
* **`GSE149980` & `GSE83386`:** Zero samples were flagged ($0/68$ and $0/20$), confirming consistent array hybridization across LCL assays.
*(Flagged QC status is preserved in exported sample QC tables so Phase 2 sensitivity analyses can evaluate inclusion vs. exclusion without irreversible data loss.)*

## 4. Probe-to-Gene Mapping & Variance Filtering
Map Illumina (`Symbol`) and Agilent (`GENE_SYMBOL`) probes to official HUGO Gene Symbols, filtering control probes and collapsing multi-probe genes by median expression. We also benchmark top-50% variance filtering per cohort.

In [ ]:
gene_exprs = {}
mapping_rows = []

for gse_id, df_n in norm_exprs.items():
    gpl_id = COHORTS[gse_id]["gpl"]
    df_gene = map_probes_to_genes(df_n, gpl_paths[gpl_id])
    df_var50 = filter_low_variance_probes(df_gene, top_percentile=0.5)
    gene_exprs[gse_id] = df_gene
    mapping_rows.append({
        "Cohort": gse_id,
        "Platform": gpl_id,
        "Normalized_Probes": df_n.shape[0],
        "Unique_Genes": df_gene.shape[0],
        "Top50pct_HighVar_Genes": df_var50.shape[0],
    })

display(pd.DataFrame(mapping_rows))

### Probe-to-Gene Mapping Analysis
* `GSE60190` (`GPL6947`): 36,068 annotated probes collapsed to **25,123 unique HUGO gene symbols** (`12,562` high-variance features at the top-50% cutoff).
* `GSE149980`, `GSE83386`, and `GSE76826` (`GPL17077`): 46,204 non-control probes collapsed to **32,080 unique HUGO gene symbols** (`16,040` high-variance features at the top-50% cutoff).

## 5. Cross-Cohort Harmonization & PGC Candidate Gene Verification
Intersect shared gene symbols across all four cohorts, evaluate pre- vs. post-Z-score batch effects using 2D PCA, and inspect baseline expression separation of PGC/SSRI candidate genes in `GSE149980`.

In [ ]:
harmonized_raw, common_genes = harmonize_cohorts(gene_exprs, zscore=False)
harmonized_z, _ = harmonize_cohorts(gene_exprs, zscore=True)

def build_pca_df(expr_dict):
    combined = pd.concat(expr_dict.values(), axis=1)
    cohort_labels = [name for name, df in expr_dict.items() for _ in range(df.shape[1])]
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(combined.T)
    return pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1], "Cohort": cohort_labels}), pca.explained_variance_ratio_

df_pca_raw, var_raw = build_pca_df(harmonized_raw)
df_pca_z, var_z = build_pca_df(harmonized_z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.scatterplot(data=df_pca_raw, x="PC1", y="PC2", hue="Cohort", ax=axes[0], s=45, alpha=0.8)
axes[0].set_title(f"Pre-Standardization PCA ({len(common_genes)} Shared Genes)")
axes[0].set_xlabel(f"PC1 ({var_raw[0]*100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({var_raw[1]*100:.1f}% variance)")

sns.scatterplot(data=df_pca_z, x="PC1", y="PC2", hue="Cohort", ax=axes[1], s=45, alpha=0.8)
axes[1].set_title("Post Z-Score Harmonization PCA")
axes[1].set_xlabel(f"PC1 ({var_z[0]*100:.1f}% variance)")
axes[1].set_ylabel(f"PC2 ({var_z[1]*100:.1f}% variance)")

plt.tight_layout()
fig_pca_path = os.path.join(FIGURES_DIR, "m1_pca_cohorts.png")
plt.savefig(fig_pca_path, dpi=150)
plt.show()

# PGC & SSRI candidate gene presence and baseline separation in GSE149980
PGC_CANDIDATES = ["SLC1A1", "SLC6A4", "HTR2A", "BDNF", "COMT", "DLGAP3", "IL6", "GRIN2B", "HTR1A", "TPH2", "MAOA"]
present_pgc = [g for g in PGC_CANDIDATES if g in common_genes]
missing_pgc = [g for g in PGC_CANDIDATES if g not in common_genes]
print(f"PGC/SSRI candidate genes in harmonized intersection ({len(present_pgc)}/{len(PGC_CANDIDATES)}): {present_pgc}")
print(f"Missing candidates: {missing_pgc}")

df_149_pgc = harmonized_z["GSE149980"].loc[present_pgc].T
df_149_pgc["Response"] = metas["GSE149980"].loc[df_149_pgc.index, "response status"]
group_means = df_149_pgc.groupby("Response").mean().T

plt.figure(figsize=(8, 6))
sns.heatmap(group_means, annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=0.5)
plt.title("GSE149980 (Citalopram): PGC & SSRI Candidate Genes (Mean Z-Score)")
plt.tight_layout()
fig_pgc_path = os.path.join(FIGURES_DIR, "m1_candidate_genes_heatmap.png")
plt.savefig(fig_pgc_path, dpi=150)
plt.show()

### Cross-Cohort Harmonization & Candidate Biomarker Analysis
* **Batch Offset Removal:** Before standardization, PC1 explained **89.9%** of variance (and PC2 **8.0%**), reflecting platform and scale differences between `GPL6947` and `GPL17077`. After per-cohort Z-score standardization across **17,188 shared genes**, PC1 dropped to **7.4%** and PC2 to **6.5%**, showing cohorts centered in a common latent space.
* **PGC & SSRI Candidate Coverage:** All **11/11** targeted candidate genes (*SLC1A1*, *SLC6A4*, *HTR2A*, *BDNF*, *COMT*, *DLGAP3*/*SAPAP3*, *IL6*, *GRIN2B*, *HTR1A*, *TPH2*, *MAOA*) are preserved in the harmonized intersection.
* **Directional Signal in `GSE149980`:** *SLC6A4* (Responder $+0.311$ vs. Non-Responder $-0.350$), *DLGAP3* ($+0.266$ vs. $-0.300$), *HTR1A* ($+0.194$ vs. $-0.218$), and *MAOA* ($-0.180$ vs. $+0.202$) show noticeable baseline mean Z-score separation prior to formal feature selection.

## 6. Export Harmonized Datasets & Metadata
Save gene-level expression matrices, harmonized Z-score matrices, parsed clinical metadata, and sample QC flags to `data/processed/` and `data/metadata/`.

In [ ]:
for gse_id in COHORTS:
    gene_exprs[gse_id].to_csv(os.path.join(DATA_PROCESSED_DIR, f"{gse_id}_gene_expr.csv.gz"), compression="gzip")
    harmonized_z[gse_id].to_csv(os.path.join(DATA_PROCESSED_DIR, f"{gse_id}_harmonized_zscore.csv.gz"), compression="gzip")
    metas[gse_id].to_csv(os.path.join(DATA_META_DIR, f"{gse_id}_metadata.csv"))
    qc_tables[gse_id].to_csv(os.path.join(DATA_META_DIR, f"{gse_id}_sample_qc.csv"))

print("Exported processed matrices, metadata, and sample QC tables successfully:")
print("  Processed:", sorted(os.listdir(DATA_PROCESSED_DIR)))
print("  Metadata:", sorted(os.listdir(DATA_META_DIR)))

### Data Analysis Key Findings
* **Cohort Verification & Replacement:** Replaced `GSE25974` (mouse embryonic tissue) and `GSE49667` (Treg DNA methylation) with verified *Homo sapiens* SSRI response cohorts `GSE149980` ($n=68$, citalopram) and `GSE83386` ($n=20$, fluoxetine), alongside `GSE60190` ($n=133$, OCD DLPFC) and `GSE76826` ($n=32$, MDD blood remission).
* **Normalization & Sample QC:** `GSE83386` raw intensities ($\text{Max}=827{,}061.30$) were $\log_2$-transformed while the remaining three cohorts were verified as already $\log_2$-scaled. Six total inter-sample correlation outliers were flagged ($4$ in `GSE60190`, $2$ in `GSE76826`, $0$ in `GSE149980`/`GSE83386`).
* **Harmonized Feature Space:** Cross-platform probe-to-gene mapping yielded **17,188 shared HUGO gene symbols** across all four cohorts with **11/11** PGC/SSRI candidate genes retained, reducing cross-cohort PC1 batch variance from **89.9%** to **7.4%**.

### Insights or Next Steps
* **Leakage-Free Cross-Validation (Phase 2):** Because `GSE149980` (17 donors across 68 arrays) and `GSE83386` (10 cell lines across 20 arrays) contain repeated-donor measurements, `src/feature_selection.py` and `src/models.py` should use `StratifiedGroupKFold` grouped by donor ID.
* **Next Milestone:** Execute this notebook in Google Colab (or locally) to review inline plots, then build `src/feature_selection.py` (ElasticNet/LASSO + pathway prioritization) to narrow the 17,188 harmonized genes down to the target **10–15 gene biomarker panel**.